# LoL Live Win Predictor – čištění dat a trénink modelu

Tento notebook popisuje celý proces:
- načtení datasetu
- základní validaci
- čištění dat
- přípravu datasetu pro live model
- trénink a vyhodnocení modelu

Notebook je určen jako dokumentace procesu tvorby modelu pro školní projekt.

## 1. Import knihoven

In [ ]:
from pathlib import Path
import json

import display
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

## 2. Nastavení cest

In [ ]:
BASE_DIR = Path('.')
RAW_DATASET_PATH = BASE_DIR / 'data' / 'processed' / 'dataset.csv'
CLEAN_DATASET_PATH = BASE_DIR / 'data' / 'processed' / 'dataset_clean.csv'
TRAIN_READY_PATH = BASE_DIR / 'data' / 'processed' / 'dataset_train_ready.csv'
LIVE_READY_PATH = BASE_DIR / 'data' / 'processed' / 'dataset_live_ready.csv'
MODEL_DIR = BASE_DIR / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('RAW_DATASET_PATH =', RAW_DATASET_PATH)
print('TRAIN_READY_PATH =', TRAIN_READY_PATH)
print('LIVE_READY_PATH =', LIVE_READY_PATH)

## 3. Načtení a základní validace datasetu

In [ ]:
df_raw = pd.read_csv(RAW_DATASET_PATH)

print('Počet řádků:', len(df_raw))
print('Počet sloupců:', len(df_raw.columns))
print('\nSloupce:')
print(df_raw.columns.tolist())

print('\nPrvních 5 řádků:')
display(df_raw.head())

if 'match_id' in df_raw.columns:
    print('\nPočet duplicitních match_id:', df_raw.duplicated(subset=['match_id']).sum())

if 'blue_win' in df_raw.columns:
    print('\nRozložení targetu blue_win:')
    print(df_raw['blue_win'].value_counts(dropna=False))

print('\nChybějící hodnoty:')
display(df_raw.isna().sum().sort_values(ascending=False).head(30))

## 4. Čištění datasetu

Použijeme jen sloupce potřebné pro trénink modelu.

In [ ]:
REQUIRED_COLUMNS = [
    'match_id',
    'blue_win',
    'blue_top_champion_name',
    'blue_jungle_champion_name',
    'blue_mid_champion_name',
    'blue_adc_champion_name',
    'blue_support_champion_name',
    'red_top_champion_name',
    'red_jungle_champion_name',
    'red_mid_champion_name',
    'red_adc_champion_name',
    'red_support_champion_name',
    'blue_avg_rank',
    'red_avg_rank',
    'blue_avg_recent_wr',
    'red_avg_recent_wr',
    'blue_avg_mastery',
    'red_avg_mastery',
]

missing_required = [col for col in REQUIRED_COLUMNS if col not in df_raw.columns]
if missing_required:
    raise ValueError(f'V datasetu chybí povinné sloupce: {missing_required}')

df_clean = df_raw[REQUIRED_COLUMNS].copy()
df_clean = df_clean.drop_duplicates(subset=['match_id']).reset_index(drop=True)
df_clean = df_clean.dropna(subset=['blue_win']).copy()
df_clean['blue_win'] = df_clean['blue_win'].astype(int)

champion_columns = [
    'blue_top_champion_name', 'blue_jungle_champion_name', 'blue_mid_champion_name',
    'blue_adc_champion_name', 'blue_support_champion_name',
    'red_top_champion_name', 'red_jungle_champion_name', 'red_mid_champion_name',
    'red_adc_champion_name', 'red_support_champion_name',
]

for col in champion_columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

numeric_columns = [
    'blue_avg_rank', 'red_avg_rank',
    'blue_avg_recent_wr', 'red_avg_recent_wr',
    'blue_avg_mastery', 'red_avg_mastery',
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

CLEAN_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_DATASET_PATH, index=False, encoding='utf-8')
df_clean.to_csv(TRAIN_READY_PATH, index=False, encoding='utf-8')

print('Počet řádků po čištění:', len(df_clean))
print('Uloženo do:', CLEAN_DATASET_PATH)
print('Uloženo do:', TRAIN_READY_PATH)
display(df_clean.head())

## 5. Příprava datasetu pro live model bez rolí

Protože spectator API negarantuje role, vytvoříme pro live model roli-nezávislý dataset.

In [ ]:
df_live = pd.DataFrame()

df_live['blue_champ_1'] = df_clean['blue_top_champion_name']
df_live['blue_champ_2'] = df_clean['blue_jungle_champion_name']
df_live['blue_champ_3'] = df_clean['blue_mid_champion_name']
df_live['blue_champ_4'] = df_clean['blue_adc_champion_name']
df_live['blue_champ_5'] = df_clean['blue_support_champion_name']

df_live['red_champ_1'] = df_clean['red_top_champion_name']
df_live['red_champ_2'] = df_clean['red_jungle_champion_name']
df_live['red_champ_3'] = df_clean['red_mid_champion_name']
df_live['red_champ_4'] = df_clean['red_adc_champion_name']
df_live['red_champ_5'] = df_clean['red_support_champion_name']

df_live['blue_avg_rank'] = df_clean['blue_avg_rank']
df_live['red_avg_rank'] = df_clean['red_avg_rank']
df_live['blue_avg_mastery'] = df_clean['blue_avg_mastery']
df_live['red_avg_mastery'] = df_clean['red_avg_mastery']
df_live['blue_avg_recent_wr'] = df_clean['blue_avg_recent_wr']
df_live['red_avg_recent_wr'] = df_clean['red_avg_recent_wr']
df_live['blue_win'] = df_clean['blue_win']

df_live.to_csv(LIVE_READY_PATH, index=False, encoding='utf-8')

print('Uloženo do:', LIVE_READY_PATH)
print('Počet řádků:', len(df_live))
display(df_live.head())

## 6. Trénink live modelu

In [ ]:
TARGET = 'blue_win'

CATEGORICAL = [
    'blue_champ_1', 'blue_champ_2', 'blue_champ_3', 'blue_champ_4', 'blue_champ_5',
    'red_champ_1', 'red_champ_2', 'red_champ_3', 'red_champ_4', 'red_champ_5',
]

NUMERIC = [
    'blue_avg_rank', 'red_avg_rank',
    'blue_avg_mastery', 'red_avg_mastery',
    'blue_avg_recent_wr', 'red_avg_recent_wr',
]

X = df_live[CATEGORICAL + NUMERIC].copy()
y = df_live[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipe, CATEGORICAL),
    ('num', num_pipe, NUMERIC),
])

models = {
    'logreg': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000))
    ]),
    'random_forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
    ]),
}

results = {}
best_name = None
best_model = None
best_f1 = -1.0

for name, model in models.items():
    print(f'Trénuji: {name}')
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    metrics = {
        'accuracy': float(accuracy_score(y_test, pred)),
        'f1': float(f1_score(y_test, pred)),
        'confusion_matrix': confusion_matrix(y_test, pred).tolist(),
        'classification_report': classification_report(y_test, pred, output_dict=True),
    }

    results[name] = metrics
    print(f"{name}: accuracy={metrics['accuracy']:.4f}, f1={metrics['f1']:.4f}")

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_name = name
        best_model = model

print('\nNejlepší model:', best_name)

## 7. Uložení modelu a metrik

In [ ]:
import joblib

MODEL_PATH = MODEL_DIR / 'live_model.pkl'
METRICS_PATH = MODEL_DIR / 'live_model_metrics.json'
FEATURE_CONFIG_PATH = MODEL_DIR / 'live_model_feature_config.json'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)

metrics_output = {
    'best_model': best_name,
    'dataset_size': int(len(df_live)),
    'train_size': int(len(X_train)),
    'test_size': int(len(X_test)),
    'results': results,
}

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_output, f, ensure_ascii=False, indent=2)

feature_config = {
    'target_column': TARGET,
    'categorical_features': CATEGORICAL,
    'numeric_features': NUMERIC,
    'all_features': CATEGORICAL + NUMERIC,
}

with open(FEATURE_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

print('Model uložen do:', MODEL_PATH)
print('Metriky uloženy do:', METRICS_PATH)
print('Feature config uložen do:', FEATURE_CONFIG_PATH)

## 8. Závěr

V tomto notebooku byl popsán celý proces od načtení původního datasetu až po natrénování live modelu.

Použitý live model je záměrně jednodušší než původní role-based model, protože live spectator data negarantují spolehlivé role hráčů. Z tohoto důvodu byl pro live predikci vytvořen role-agnostic model pracující pouze s výběrem championů a agregovanými týmovými statistikami.